### Multiconductor & OpenDSS Load Flow Results

In [ ]:
def export_dss_loadflow_results(circuit_name, dss_file):
    """Run OpenDSS power flow on a .dss file and export per-bus results to CSV.
    Returns a DataFrame of the results."""
    import csv
    import os
    import math
    import numpy as np
    import pandas as pd

    # Pre-load the correct KLUSolve.dll from py_dss_interface's own directory
    # to prevent Windows from picking up the incompatible copy in the workspace root.
    import ctypes
    _pdi_dll_dir = os.path.join(
        os.path.dirname(__import__('py_dss_interface').__file__),
        'opendss_official', 'windows', 'delphi', 'x64',
    )
    _klu_path = os.path.join(_pdi_dll_dir, 'KLUSolve.dll')
    if os.path.isfile(_klu_path):
        if hasattr(os, 'add_dll_directory'):
            os.add_dll_directory(_pdi_dll_dir)
        ctypes.WinDLL(_klu_path)

    from opendss.pf.powerflow import run_pf as dss_run_pf

    results_csv = os.path.join(r"c:\Users\fgonzales\git\mc-0.0.1.15", f"{circuit_name}_dss_loadflow_results.csv")

    # Run OpenDSS power flow
    pf = dss_run_pf(str(dss_file))
    d = pf['dss']

    if not pf['converged']:
        print(f"WARNING: DSS power flow did not converge for {circuit_name}")

    PRECISION = 7

    # ---- Build per-bus power maps from element iteration ----
    def _collect_element_powers(element_class, element_prefix):
        """Iterate DSS elements, return {bus_name_lower: {phase: [p_kw, q_kvar]}}."""
        power_map = {}
        elem_api = getattr(d, element_class, None)
        if elem_api is None:
            return power_map
        try:
            i = elem_api.first()
        except Exception:
            return power_map
        while i != 0:
            ename = elem_api.name
            d.circuit.set_active_element(f"{element_prefix}.{ename}")
            bus_names = d.cktelement.bus_names
            powers = d.cktelement.powers          # [P1, Q1, P2, Q2, ...]
            n = d.cktelement.num_conductors
            if bus_names:
                parts = bus_names[0].split('.')
                bname = parts[0].lower()
                phases = [int(p) for p in parts[1:]] if len(parts) > 1 else list(range(1, n + 1))
                if bname not in power_map:
                    power_map[bname] = {}
                for j, ph in enumerate(phases):
                    if j * 2 + 1 < len(powers):
                        if ph not in power_map[bname]:
                            power_map[bname][ph] = [0.0, 0.0]
                        power_map[bname][ph][0] += powers[j * 2]
                        power_map[bname][ph][1] += powers[j * 2 + 1]
            i = elem_api.next()
        return power_map

    load_map = _collect_element_powers('loads', 'Load')
    gen_map  = _collect_element_powers('generators', 'Generator')
    cap_map  = _collect_element_powers('capacitors', 'Capacitor')

    # Phase number → letter mapping
    PHASE_LETTER = {1: 'A', 2: 'B', 3: 'C'}

    num_buses = d.circuit.num_buses
    rows = []

    for i in range(num_buses):
        d.circuit.set_active_bus_i(i)
        name   = d.bus.name
        kv_base = d.bus.kv_base              # L-N kV
        nodes  = list(d.bus.nodes)            # e.g. [1, 2, 3]
        va_pu  = d.bus.vmag_angle_pu          # [mag1, ang1, mag2, ang2, ...]

        # Per-phase voltage magnitudes (pu) and angles (deg)
        v_pu  = {}
        v_ang = {}
        for j, node in enumerate(nodes):
            if j * 2 + 1 < len(va_pu):
                v_pu[node]  = round(va_pu[j * 2],     PRECISION)
                v_ang[node] = round(va_pu[j * 2 + 1], PRECISION)

        # Total per-bus power (sum across phases), kW → MW
        name_lower = name.lower()
        lp = load_map.get(name_lower, {})
        gp = gen_map.get(name_lower, {})
        cp = cap_map.get(name_lower, {})

        pload = round(sum(v[0] for v in lp.values()) / 1000.0, PRECISION)
        qload = round(sum(v[1] for v in lp.values()) / 1000.0, PRECISION)
        pgen  = round(sum(v[0] for v in gp.values()) / 1000.0, PRECISION)
        qgen  = round(sum(v[1] for v in gp.values()) / 1000.0, PRECISION)
        pcap  = round(sum(v[0] for v in cp.values()) / 1000.0, PRECISION)
        qcap  = round(sum(v[1] for v in cp.values()) / 1000.0, PRECISION)

        # Per-phase load and gen power (kW → MW, kVAR → MVAR)
        per_phase_load = {}
        per_phase_gen = {}
        for ph, letter in PHASE_LETTER.items():
            lph = lp.get(ph, [0.0, 0.0])
            gph = gp.get(ph, [0.0, 0.0])
            per_phase_load[f'PLOAD_{letter}'] = round(lph[0] / 1000.0, PRECISION)
            per_phase_load[f'QLOAD_{letter}'] = round(lph[1] / 1000.0, PRECISION)
            per_phase_gen[f'PGEN_{letter}']   = round(gph[0] / 1000.0, PRECISION)
            per_phase_gen[f'QGEN_{letter}']   = round(gph[1] / 1000.0, PRECISION)

        # Per-phase injection currents: I = conj(S / V)
        i_mag = {}
        i_ang_d = {}
        for phase in [1, 2, 3]:
            if phase in v_pu and v_pu[phase] > 0 and kv_base > 0:
                v_actual_v = v_pu[phase] * kv_base * 1000.0
                v_rad = np.deg2rad(v_ang[phase])
                V_c = v_actual_v * np.exp(1j * v_rad)
                p_w  = ((gp.get(phase, [0, 0])[0] - lp.get(phase, [0, 0])[0]
                         + cp.get(phase, [0, 0])[0]) * 1000.0)
                q_va = ((gp.get(phase, [0, 0])[1] - lp.get(phase, [0, 0])[1]
                         + cp.get(phase, [0, 0])[1]) * 1000.0)
                if abs(V_c) > 0:
                    I_c = np.conj((p_w + 1j * q_va) / V_c)
                    i_mag[phase]  = round(float(abs(I_c)), PRECISION)
                    i_ang_d[phase] = round(float(np.rad2deg(np.angle(I_c))), PRECISION)

        kv_ll = round(kv_base * math.sqrt(3), PRECISION) if kv_base > 0 else 0

        row = {
            'CKT_KEY': circuit_name,
            'NODE_ID': name,
            'BASE_VOLTAGE': kv_ll,
            'PGEN': pgen, 'QGEN': qgen,
            'PGEN_A': per_phase_gen['PGEN_A'], 'PGEN_B': per_phase_gen['PGEN_B'], 'PGEN_C': per_phase_gen['PGEN_C'],
            'QGEN_A': per_phase_gen['QGEN_A'], 'QGEN_B': per_phase_gen['QGEN_B'], 'QGEN_C': per_phase_gen['QGEN_C'],
            'PCAP': pcap, 'QCAP': qcap,
            'PLOAD': pload, 'QLOAD': qload,
            'PLOAD_A': per_phase_load['PLOAD_A'], 'PLOAD_B': per_phase_load['PLOAD_B'], 'PLOAD_C': per_phase_load['PLOAD_C'],
            'QLOAD_A': per_phase_load['QLOAD_A'], 'QLOAD_B': per_phase_load['QLOAD_B'], 'QLOAD_C': per_phase_load['QLOAD_C'],
            'VA': v_pu.get(1, None),  'VA_ANGLE': v_ang.get(1, None),
            'VB': v_pu.get(2, None),  'VB_ANGLE': v_ang.get(2, None),
            'VC': v_pu.get(3, None),  'VC_ANGLE': v_ang.get(3, None),
            'IA': i_mag.get(1, None), 'IA_ANGLE': i_ang_d.get(1, None),
            'IB': i_mag.get(2, None), 'IB_ANGLE': i_ang_d.get(2, None),
            'IC': i_mag.get(3, None), 'IC_ANGLE': i_ang_d.get(3, None),
        }
        rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(results_csv, index=False)
    print(f"Wrote {num_buses} buses to {results_csv}")
    return df

In [ ]:
def export_mc_loadflow_results(net, circuit_name):
    """Run multiconductor CCI power flow on *net* and export per-bus results to CSV.
    Returns a DataFrame of the results."""
    import csv
    import os
    import numpy as np
    import pandas as pd
    from multiconductor.pycci.cci_powerflow import run_pf

    results_csv = os.path.join(r"c:\Users\fgonzales\git\mc-0.0.1.15", f"{circuit_name}_mc_loadflow_results.csv")

    # Run CCI power flow (populates net.res_bus, res_asymmetric_*, etc.)
    run_pf(net)

    PRECISION = 7
    bus_indices = net.bus.index.get_level_values(0).unique()

    # Phase number → letter mapping
    PHASE_LETTER = {1: 'A', 2: 'B', 3: 'C'}

    # ---- Aggregate element power per bus (total and per-phase) ----
    def _aggregate_power(element_name):
        """Return {bus_idx: {'total': [p, q], 'A': [p,q], 'B': [p,q], 'C': [p,q]}}."""
        elem_key = f"asymmetric_{element_name}"
        res_key  = f"res_asymmetric_{element_name}"
        result = {}
        if elem_key not in net or res_key not in net:
            return result
        elems = net[elem_key]
        res   = net[res_key]
        if elems.empty or res.empty:
            return result
        for idx in res.index:
            bus = int(elems.loc[idx, 'bus'])
            p = float(res.loc[idx, 'p_mw'])
            q = float(res.loc[idx, 'q_mvar'])
            from_phase = int(elems.loc[idx, 'from_phase'])
            letter = PHASE_LETTER.get(from_phase, None)
            if bus not in result:
                result[bus] = {'total': [0.0, 0.0], 'A': [0.0, 0.0], 'B': [0.0, 0.0], 'C': [0.0, 0.0]}
            result[bus]['total'][0] += p
            result[bus]['total'][1] += q
            if letter:
                result[bus][letter][0] += p
                result[bus][letter][1] += q
        return result

    load_power  = _aggregate_power('load')
    sgen_power  = _aggregate_power('sgen')
    shunt_power = _aggregate_power('shunt')

    rows = []
    for bus_idx in bus_indices:
        bus_data = net.bus.loc[bus_idx]
        if isinstance(bus_data, pd.DataFrame):
            name  = bus_data['name'].iloc[0]
            vn_kv = float(bus_data['vn_kv'].iloc[0])
        else:
            name  = bus_data['name']
            vn_kv = float(bus_data['vn_kv'])

        # Per-phase voltages from res_bus (MultiIndex: bus_idx, phase)
        v_pu  = {}
        v_ang = {}
        for phase in [1, 2, 3]:
            if (bus_idx, phase) in net.res_bus.index:
                vm = float(net.res_bus.loc[(bus_idx, phase), 'vm_pu'])
                va = float(net.res_bus.loc[(bus_idx, phase), 'va_degree'])
                if not np.isnan(vm):
                    v_pu[phase]  = round(vm, PRECISION)
                    v_ang[phase] = round(va, PRECISION)

        # Power at bus (total and per-phase)
        _empty = {'total': [0.0, 0.0], 'A': [0.0, 0.0], 'B': [0.0, 0.0], 'C': [0.0, 0.0]}
        lp = load_power.get(bus_idx, _empty)
        gp = sgen_power.get(bus_idx, _empty)
        sp = shunt_power.get(bus_idx, _empty)

        # Per-phase injection currents: I = conj(S / V)
        kv_ln = vn_kv / np.sqrt(3) if vn_kv > 0 else 0
        i_mag = {}
        i_ang_d = {}
        for phase in [1, 2, 3]:
            if phase in v_pu and v_pu[phase] > 0 and kv_ln > 0:
                if (bus_idx, phase) in net.res_bus.index:
                    p_mw   = float(net.res_bus.loc[(bus_idx, phase), 'p_mw'])
                    q_mvar = float(net.res_bus.loc[(bus_idx, phase), 'q_mvar'])
                    v_actual_v = v_pu[phase] * kv_ln * 1000.0
                    v_rad = np.deg2rad(v_ang[phase])
                    V_c = v_actual_v * np.exp(1j * v_rad)
                    if abs(V_c) > 0:
                        S = (p_mw * 1e6 + 1j * q_mvar * 1e6)
                        I_c = np.conj(S / V_c)
                        i_mag[phase]  = round(float(abs(I_c)), PRECISION)
                        i_ang_d[phase] = round(float(np.rad2deg(np.angle(I_c))), PRECISION)

        rows.append({
            'CKT_KEY': circuit_name,
            'NODE_ID': name,
            'BASE_VOLTAGE': round(vn_kv, PRECISION),
            'PGEN': round(gp['total'][0], PRECISION), 'QGEN': round(gp['total'][1], PRECISION),
            'PGEN_A': round(gp['A'][0], PRECISION), 'PGEN_B': round(gp['B'][0], PRECISION), 'PGEN_C': round(gp['C'][0], PRECISION),
            'QGEN_A': round(gp['A'][1], PRECISION), 'QGEN_B': round(gp['B'][1], PRECISION), 'QGEN_C': round(gp['C'][1], PRECISION),
            'PCAP': round(sp['total'][0], PRECISION), 'QCAP': round(sp['total'][1], PRECISION),
            'PLOAD': round(lp['total'][0], PRECISION), 'QLOAD': round(lp['total'][1], PRECISION),
            'PLOAD_A': round(lp['A'][0], PRECISION), 'PLOAD_B': round(lp['B'][0], PRECISION), 'PLOAD_C': round(lp['C'][0], PRECISION),
            'QLOAD_A': round(lp['A'][1], PRECISION), 'QLOAD_B': round(lp['B'][1], PRECISION), 'QLOAD_C': round(lp['C'][1], PRECISION),
            'VA': v_pu.get(1, None),  'VA_ANGLE': v_ang.get(1, None),
            'VB': v_pu.get(2, None),  'VB_ANGLE': v_ang.get(2, None),
            'VC': v_pu.get(3, None),  'VC_ANGLE': v_ang.get(3, None),
            'IA': i_mag.get(1, None), 'IA_ANGLE': i_ang_d.get(1, None),
            'IB': i_mag.get(2, None), 'IB_ANGLE': i_ang_d.get(2, None),
            'IC': i_mag.get(3, None), 'IC_ANGLE': i_ang_d.get(3, None),
        })

    df = pd.DataFrame(rows)
    df.to_csv(results_csv, index=False)
    print(f"Wrote {len(bus_indices)} buses to {results_csv}")
    return df

In [ ]:
import pickle
from pathlib import Path
import pandas as pd
import opendss.tools.dss_converter as odc
from multiconductor.pycci.cci_powerflow import run_pf


CKT_4053_04314 = 'CKT_4053_04314'
CKT_114_16955 = 'CKT_114_16955'
CKT_4799_01085 = 'CKT_4799_01085'

validation_nets = [CKT_4053_04314, CKT_114_16955, CKT_4799_01085]

all_results = {} 

for circuit_key in validation_nets:
    print(f"Processing {circuit_key}...")
    pkl_file = Path(rf"c:\Users\fgonzales\git\mc-0.0.1.15\networks_final\{circuit_key}.pkl")
    dss_file = Path(rf"c:\Users\fgonzales\git\mc-0.0.1.15\networks_final\{circuit_key}.dss")

    with open(pkl_file, 'rb') as fh:
        net = pickle.load(fh)

    df_mc = export_mc_loadflow_results(net, circuit_key)

    odc.mc_net_to_opendss(net, str(dss_file))
    df_dss = export_dss_loadflow_results(circuit_key, dss_file)

    all_results[circuit_key] = {'mc': df_mc, 'dss': df_dss, 'net': net}

print("All circuits processed.")

In [ ]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np

NUMERIC_FIELDS = [
    'BASE_VOLTAGE',
    'PGEN', 'QGEN',
    'PGEN_A', 'PGEN_B', 'PGEN_C',
    'QGEN_A', 'QGEN_B', 'QGEN_C',
    'PCAP', 'QCAP',
    'PLOAD', 'QLOAD',
    'PLOAD_A', 'PLOAD_B', 'PLOAD_C',
    'QLOAD_A', 'QLOAD_B', 'QLOAD_C',
    'VA', 'VA_ANGLE', 'VB', 'VB_ANGLE', 'VC', 'VC_ANGLE',
    'IA', 'IA_ANGLE', 'IB', 'IB_ANGLE', 'IC', 'IC_ANGLE',
]

SOURCE_COLORS = {'mc': '#1f77b4', 'dss': '#ff7f0e'}
SOURCE_SYMBOLS = {'mc': 'circle', 'dss': 'diamond'}
SOURCES = ['mc', 'dss']
circuit_keys = list(all_results.keys())

def _get_trace_data(ckt, field, label):
    """Return (x, y, text) arrays for one source trace."""
    df = all_results.get(ckt, {}).get(label)
    if df is not None and not df.empty and field in df.columns:
        col = pd.to_numeric(df[field], errors='coerce')
        mask = col.notna()
        return np.arange(mask.sum()), col[mask].values, df.loc[mask, 'NODE_ID'].values
    return np.array([]), np.array([]), np.array([])

# --- Create figure with exactly 3 traces (one per source) ---
default_ckt = circuit_keys[0]
default_field = 'VA'

fig = go.Figure()
for label in SOURCES:
    x, y, text = _get_trace_data(default_ckt, default_field, label)
    fig.add_trace(go.Scatter(
        x=x, y=y, text=text,
        mode='markers',
        name=label.upper(),
        marker=dict(color=SOURCE_COLORS[label], symbol=SOURCE_SYMBOLS[label], size=6, opacity=0.7),
        hovertemplate='%{text}<br>' + default_field + ': %{y}<extra></extra>',
    ))

# --- Circuit dropdown: updates x/y/text data on the 3 traces ---
circuit_buttons = []
for ckt in circuit_keys:
    xd, yd, td = [], [], []
    for label in SOURCES:
        x, y, text = _get_trace_data(ckt, 'VA', label)
        xd.append(x); yd.append(y); td.append(text)
    ht = ['%{text}<br>VA: %{y}<extra></extra>'] * 3
    circuit_buttons.append(dict(
        label=ckt, method='update',
        args=[{'x': xd, 'y': yd, 'text': td, 'hovertemplate': ht},
              {'title': f'{ckt} — VA', 'yaxis.title': 'VA'}],
    ))

# --- Field dropdown: updates x/y/text data on the 3 traces ---
field_buttons = []
for field in NUMERIC_FIELDS:
    xd, yd, td = [], [], []
    for label in SOURCES:
        x, y, text = _get_trace_data(default_ckt, field, label)
        xd.append(x); yd.append(y); td.append(text)
    ht = ['%{text}<br>' + field + ': %{y}<extra></extra>'] * 3
    field_buttons.append(dict(
        label=field, method='update',
        args=[{'x': xd, 'y': yd, 'text': td, 'hovertemplate': ht},
              {'title': f'{default_ckt} — {field}', 'yaxis.title': field}],
    ))

# --- Source toggle buttons ---
source_buttons = [
    dict(label='All', method='restyle',
         args=[{'visible': [True, True, True]}]),
    dict(label='MC', method='restyle',
         args=[{'visible': [True, 'legendonly', 'legendonly']}]),
    dict(label='DSS', method='restyle',
         args=[{'visible': ['legendonly', True, 'legendonly']}]),
]

fig.update_layout(
    template='plotly_dark',
    title=f'{default_ckt} — {default_field}',
    xaxis_title='Node index',
    yaxis_title=default_field,
    height=550,
    legend_title='Source',
    updatemenus=[
        dict(buttons=circuit_buttons, direction='down', showactive=True,
             x=0.0, xanchor='left', y=1.18, yanchor='top'),
        dict(buttons=field_buttons, direction='down', showactive=True,
             x=0.28, xanchor='left', y=1.18, yanchor='top'),
        dict(buttons=source_buttons, type='buttons', direction='right',
             showactive=True, active=0,
             x=0.55, xanchor='left', y=1.18, yanchor='top'),
    ],
    annotations=[
        dict(text='Circuit:', x=0.0, xref='paper', xanchor='right',
             y=1.15, yref='paper', showarrow=False, xshift=-5),
        dict(text='Field:', x=0.28, xref='paper', xanchor='right',
             y=1.15, yref='paper', showarrow=False, xshift=-5),
        dict(text='Sources:', x=0.55, xref='paper', xanchor='right',
             y=1.15, yref='paper', showarrow=False, xshift=-5),
    ],
)
fig.show()

In [ ]:
import pandas as pd
import numpy as np
import os

COMPARE_FIELDS = [
    'BASE_VOLTAGE',
    'PGEN', 'QGEN',
    'PGEN_A', 'PGEN_B', 'PGEN_C',
    'QGEN_A', 'QGEN_B', 'QGEN_C',
    'PCAP', 'QCAP',
    'PLOAD', 'QLOAD',
    'PLOAD_A', 'PLOAD_B', 'PLOAD_C',
    'QLOAD_A', 'QLOAD_B', 'QLOAD_C',
    'VA', 'VA_ANGLE', 'VB', 'VB_ANGLE', 'VC', 'VC_ANGLE',
    'IA', 'IA_ANGLE', 'IB', 'IB_ANGLE', 'IC', 'IC_ANGLE',
]

def _build_element_map(net):
    """Return {bus_name: comma-separated element types} from the net object."""
    bus_elements = {}  # bus_idx -> set of element type strings

    element_tables = [
        ('asymmetric_load',  'bus', 'load'),
        ('asymmetric_sgen',  'bus', 'sgen'),
        ('asymmetric_shunt', 'bus', 'shunt'),
    ]
    for table_name, bus_col, label in element_tables:
        if table_name in net and not net[table_name].empty:
            for bus_idx in net[table_name][bus_col].unique():
                bus_elements.setdefault(int(bus_idx), set()).add(label)

    # 2-winding transformers (trafo or trafo1ph)
    if 'trafo' in net and not net['trafo'].empty:
        for _, row in net['trafo'].iterrows():
            for col in ['hv_bus', 'lv_bus']:
                if col in row.index:
                    bus_elements.setdefault(int(row[col]), set()).add('trafo')

    # trafo1ph: bus is stored in the MultiIndex, not as a column
    if 'trafo1ph' in net and not net['trafo1ph'].empty:
        for bus_idx in net['trafo1ph'].index.get_level_values('bus').unique():
            bus_elements.setdefault(int(bus_idx), set()).add('trafo')

    # 3-winding transformers
    if 'trafo3w' in net and not net['trafo3w'].empty:
        for _, row in net['trafo3w'].iterrows():
            for col in ['hv_bus', 'mv_bus', 'lv_bus']:
                if col in row.index:
                    bus_elements.setdefault(int(row[col]), set()).add('trafo3w')

    # ext_grid (source/gen) — check both ext_grid and ext_grid_sequence
    for eg_name in ['ext_grid', 'ext_grid_sequence']:
        if eg_name in net and not net[eg_name].empty:
            if 'bus' in net[eg_name].columns:
                for bus_idx in net[eg_name]['bus'].unique():
                    bus_elements.setdefault(int(bus_idx), set()).add('ext_grid')
            elif 'bus' in net[eg_name].index.names:
                for bus_idx in net[eg_name].index.get_level_values('bus').unique():
                    bus_elements.setdefault(int(bus_idx), set()).add('ext_grid')

    # Map bus index → bus name
    name_map = {}
    for bus_idx in net.bus.index.get_level_values(0).unique():
        bus_data = net.bus.loc[bus_idx]
        name = bus_data['name'].iloc[0] if isinstance(bus_data, pd.DataFrame) else bus_data['name']
        name_map[int(bus_idx)] = name

    # Build {bus_name: element_string}
    result = {}
    for bus_idx, elem_set in bus_elements.items():
        if bus_idx in name_map:
            result[name_map[bus_idx]] = ', '.join(sorted(elem_set))
    return result

def compare_loadflow_results(df_mc, df_dss, circuit_name, df_cyme=None, net=None, output_dir=r"c:\Users\fgonzales\git\mc-0.0.1.15\networks_final\validation"):
    """Compare MC, DSS, and optionally CYME load flow results side-by-side with % difference.

    Joins on NODE_ID, then for each numeric field produces columns:
        <field>_MC, <field>_DSS, <field>_%DIFF_DSS
    and if df_cyme is provided:
        <field>_CYME, <field>_%DIFF_CYME
    where %DIFF = (MC - other) / MC * 100.  When MC == 0 the diff is NaN.

    Returns the merged DataFrame and writes it to CSV.
    """
    mc = df_mc[['NODE_ID'] + COMPARE_FIELDS].copy()
    dss = df_dss[['NODE_ID'] + COMPARE_FIELDS].copy()

    for col in COMPARE_FIELDS:
        mc[col] = pd.to_numeric(mc[col], errors='coerce')
        dss[col] = pd.to_numeric(dss[col], errors='coerce')

    # Rename columns before merge to avoid suffix collisions
    mc = mc.rename(columns={c: f'{c}_MC' for c in COMPARE_FIELDS})
    dss = dss.rename(columns={c: f'{c}_DSS' for c in COMPARE_FIELDS})

    merged = mc.merge(dss, on='NODE_ID', how='outer')

    # Compute MC vs DSS % diff
    for col in COMPARE_FIELDS:
        mc_col = f'{col}_MC'
        dss_col = f'{col}_DSS'
        diff_col = f'{col}_%DIFF_DSS'
        merged[diff_col] = np.where(
            merged[mc_col] != 0,
            (merged[mc_col] - merged[dss_col]) / merged[mc_col] * 100,
            np.nan,
        )

    # Merge CYME if provided
    if df_cyme is not None:
        cyme = df_cyme[['NODE_ID'] + COMPARE_FIELDS].copy()
        cyme['NODE_ID'] = cyme['NODE_ID'].astype(str)
        merged['NODE_ID'] = merged['NODE_ID'].astype(str)
        for col in COMPARE_FIELDS:
            cyme[col] = pd.to_numeric(cyme[col], errors='coerce')
        cyme = cyme.rename(columns={c: f'{c}_CYME' for c in COMPARE_FIELDS})
        merged = merged.merge(cyme, on='NODE_ID', how='outer')

        for col in COMPARE_FIELDS:
            mc_col = f'{col}_MC'
            cyme_col = f'{col}_CYME'
            diff_col = f'{col}_%DIFF_CYME'
            merged[diff_col] = np.where(
                merged[mc_col] != 0,
                (merged[mc_col] - merged[cyme_col]) / merged[mc_col] * 100,
                np.nan,
            )

    # Add circuit key
    merged.insert(0, 'CKT_KEY', circuit_name)

    # Add element type column from net
    if net is not None:
        elem_map = _build_element_map(net)
        merged.insert(2, 'ELEMENT', merged['NODE_ID'].map(elem_map).fillna(''))
    else:
        merged.insert(2, 'ELEMENT', '')

    # Reorder columns: CKT_KEY, NODE_ID, ELEMENT, then groups per field
    ordered_cols = ['CKT_KEY', 'NODE_ID', 'ELEMENT']
    for col in COMPARE_FIELDS:
        ordered_cols += [f'{col}_MC', f'{col}_DSS', f'{col}_%DIFF_DSS']
        if df_cyme is not None:
            ordered_cols += [f'{col}_CYME', f'{col}_%DIFF_CYME']
    merged = merged[ordered_cols]

    out_csv = os.path.join(output_dir, f"{circuit_name}_mc_vs_dss_comparison.csv")
    merged.to_csv(out_csv, index=False)
    print(f"Wrote {len(merged)} rows to {out_csv}")
    return merged

# Run comparison for all circuits
comparison_results = {}
df = pd.read_csv(rf"c:\Users\fgonzales\git\mc-0.0.1.15\scripts\encoded.csv")
for row in df.itertuples():
    ckt = row.CIRCUIT_KEY
    mc_csv = rf"c:\Users\fgonzales\git\mc-0.0.1.15\networks_final\validation\{ckt}_mc_loadflow_results.csv"
    dss_csv = rf"c:\Users\fgonzales\git\mc-0.0.1.15\networks_final\validation\{ckt}_dss_loadflow_results.csv"
    cyme_csv = rf"c:\Users\fgonzales\git\mc-0.0.1.15\networks_final\validation\{ckt}_cyme_loadflow_results.csv"

    if not os.path.isfile(mc_csv) or not os.path.isfile(dss_csv):
        print(f"Skipping {ckt} — missing MC or DSS CSV(s)")
        continue

    df_mc = pd.read_csv(mc_csv)
    df_dss = pd.read_csv(dss_csv)
    df_cyme = pd.read_csv(cyme_csv) if os.path.isfile(cyme_csv) else None

    print(f"Comparing {ckt}..." + (" (with CYME)" if df_cyme is not None else ""))
    comparison_results[ckt] = compare_loadflow_results(df_mc, df_dss, ckt, df_cyme=df_cyme)

comparison_results[list(comparison_results.keys())[0]].head(10)

In [ ]:
all_df = comparison_results[list(comparison_results.keys())[0]]
all_df

In [ ]:
all_df = comparison_results[list(comparison_results.keys())[1]]

In [ ]:
cols = ["CKT_KEY", "NODE_ID",
    "VA_MC", "VA_DSS", "VA_CYME",
    "VB_MC", "VB_DSS", "VB_CYME",
    "VC_MC", "VC_DSS", "VC_CYME"
]

new_df = all_df[cols].copy()

In [ ]:

new_df

In [ ]:
numeric_cols = new_df.select_dtypes(include='number').columns.tolist()
summary = new_df.groupby('CKT_KEY')[numeric_cols].mean()
summary

In [ ]:
## MC vs CYME: Bus Voltage & Transformer Configuration Comparison

# Compare assigned bus voltages (`vn_kv`) and transformer configurations between the multiconductor
# net objects and what CYME actually uses after the mc→CYME conversion.

# **Key differences to check:**
# 1. **Bus base voltages** — MC stores explicit `vn_kv` per bus; CYME sets `UserDefinedBaseVoltage` only on transformer LV nodes
# 2. **Transformer winding voltages** — MC converts L-L → winding voltage (÷√3 for Y, direct for D); CYME uses L-L directly
# 3. **Connection type** — MC stores full vector group (from_phase/to_phase per circuit); CYME simplifies to `D_Yg` / `Yg_Yg`
# 4. **Impedance split** — MC stores half-impedance per winding side; CYME uses total positive-sequence %Z
# 5. **Power flow settings** — CYME disables line charging and voltage-sensitive loads by default

In [ ]:
import pickle, math, os, time
from pathlib import Path
import pandas as pd
import numpy as np

PHASE_INT_TO_LETTER = {0: 'N', 1: 'A', 2: 'B', 3: 'C'}

def compare_bus_voltages_and_transformers(net, circuit_name):
    """Extract bus voltages and transformer configs from an mc net object
    and compute the equivalent CYME representation side-by-side."""

    # ------------------------------------------------------------------ #
    # PRE-COMPUTE LOOKUPS (avoid repeated DataFrame operations)           #
    # ------------------------------------------------------------------ #

    # -- Source bus lookup --
    source_buses = {}  # {bus_idx: vm_pu}
    for eg_name in ['ext_grid_sequence', 'ext_grid']:
        if eg_name in net and not net[eg_name].empty:
            eg = net[eg_name]
            if 'bus' in eg.columns:
                for bus_idx in eg['bus'].unique():
                    mask = eg['bus'] == bus_idx
                    source_buses[int(bus_idx)] = float(eg.loc[mask, 'vm_pu'].iloc[0])
            elif 'bus' in eg.index.names:
                level = eg.index.names.index('bus')
                for bus_idx in eg.index.get_level_values(level).unique():
                    source_buses[int(bus_idx)] = 1.0

    # -- Transformer LV bus lookup (built once, not per-bus) --
    trafo_lv_map = {}  # {lv_bus_idx: trafo_name}
    trafo_flat_cache = {}  # {trafo_idx: (flat_df, buses)}
    if 'trafo1ph' in net and not net['trafo1ph'].empty:
        trafo_indices = net.trafo1ph.index.get_level_values(0).unique()
        for t_idx in trafo_indices:
            t_data = net.trafo1ph.loc[t_idx]
            if isinstance(t_data, pd.Series):
                t_data = t_data.to_frame().T
            flat = t_data.reset_index() if isinstance(t_data.index, pd.MultiIndex) else t_data
            if 'bus' in flat.columns:
                buses = sorted(set(flat['bus'].values))
            else:
                buses = sorted(set(flat.index.get_level_values('bus')))
            trafo_flat_cache[t_idx] = (flat, buses)
            if len(buses) >= 2:
                trafo_lv_map[buses[1]] = f'trafo_{t_idx}'

    # -- Bus name/vn_kv lookup (vectorized) --
    bus_indices = net.bus.index.get_level_values(0).unique()
    bus_phase_idx = net.bus.index.get_level_values(-1)

    # ------------------------------------------------------------------ #
    # 1. BUS VOLTAGE TABLE                                                #
    # ------------------------------------------------------------------ #
    bus_rows = []
    for bus_idx in bus_indices:
        bus_data = net.bus.loc[bus_idx]
        if isinstance(bus_data, pd.DataFrame):
            name = bus_data['name'].iloc[0]
            vn_kv = float(bus_data['vn_kv'].iloc[0])
            phases = sorted(set(bus_data.index.get_level_values(-1)) - {0})
        else:
            name = bus_data['name']
            vn_kv = float(bus_data['vn_kv'])
            phases = []

        phase_str = ''.join(PHASE_INT_TO_LETTER.get(p, '') for p in phases)
        is_source = bus_idx in source_buses
        vm_pu = source_buses.get(bus_idx, 1.0)
        is_trafo_lv = bus_idx in trafo_lv_map
        trafo_name = trafo_lv_map.get(bus_idx, '')

        cyme_base_voltage = vn_kv if (is_trafo_lv or is_source) else 'inherited/unset'
        cyme_operating_kv = round(vn_kv * vm_pu / math.sqrt(3), 6) if is_source else ''

        bus_rows.append({
            'CKT_KEY': circuit_name,
            'BUS_IDX': bus_idx,
            'BUS_NAME': name,
            'PHASES': phase_str,
            'MC_VN_KV (L-L)': vn_kv,
            'IS_SOURCE': is_source,
            'IS_TRAFO_LV': is_trafo_lv,
            'TRAFO_NAME': trafo_name,
            'CYME_UserDefinedBaseVoltage': cyme_base_voltage,
            'CYME_OperatingVoltage_LN': cyme_operating_kv,
            'VM_PU': vm_pu if is_source else '',
        })

    df_buses = pd.DataFrame(bus_rows)

    # ------------------------------------------------------------------ #
    # 2. TRANSFORMER TABLE (uses cached flat DataFrames)                  #
    # ------------------------------------------------------------------ #
    trafo_rows = []
    for t_idx, (flat, buses) in trafo_flat_cache.items():
        hv_bus = buses[0]
        lv_bus = buses[1] if len(buses) > 1 else buses[0]

        first = flat.iloc[0]
        sn_mva = float(first.get('sn_mva', 0))
        vk_pct = float(first.get('vk_percent', 0))
        vkr_pct = float(first.get('vkr_percent', 0))
        tap_pos = float(first.get('tap_pos', 0))

        hv_vn_kv = float(net.bus.loc[(hv_bus, 1)]['vn_kv']) if (hv_bus, 1) in net.bus.index else float(net.bus.loc[hv_bus].iloc[0]['vn_kv'])
        lv_vn_kv = float(net.bus.loc[(lv_bus, 1)]['vn_kv']) if (lv_bus, 1) in net.bus.index else float(net.bus.loc[lv_bus].iloc[0]['vn_kv'])

        mc_vn_kv_list = flat['vn_kv'].values.tolist()
        from_phases = flat['from_phase'].values.tolist()
        to_phases = flat['to_phase'].values.tolist()

        phase_letters = set()
        for fp in from_phases:
            p = PHASE_INT_TO_LETTER.get(int(fp), '')
            if p and p != 'N':
                phase_letters.add(p)
        phase_str = ''.join(sorted(phase_letters))

        tp_set = set(int(x) for x in to_phases)
        has_delta = any(tp != 0 for tp in tp_set)
        cyme_connection = 'D_Yg' if has_delta else 'Yg_Yg'

        cyme_z1_pct = vk_pct
        cyme_x1r1 = round(math.sqrt(max(0, (vk_pct / vkr_pct) ** 2 - 1)), 4) if vkr_pct > 0 else 10.0
        cyme_rating_kva = sn_mva * 1000
        mc_total_sn_mva = sn_mva * len(flat) / len(buses)

        hv_winding = mc_vn_kv_list[0] if mc_vn_kv_list else 0
        lv_winding = mc_vn_kv_list[len(flat)//2] if len(mc_vn_kv_list) > len(flat)//2 else (mc_vn_kv_list[-1] if mc_vn_kv_list else 0)

        hv_is_y = abs(hv_winding - hv_vn_kv / math.sqrt(3)) < 0.01 if hv_winding > 0 else False
        hv_is_d = abs(hv_winding - hv_vn_kv) < 0.01 if hv_winding > 0 else False
        lv_is_y = abs(lv_winding - lv_vn_kv / math.sqrt(3)) < 0.01 if lv_winding > 0 else False
        lv_is_d = abs(lv_winding - lv_vn_kv) < 0.01 if lv_winding > 0 else False

        hv_conn_type = 'Y' if hv_is_y else ('D' if hv_is_d else '?')
        lv_conn_type = 'Y' if lv_is_y else ('D' if lv_is_d else '?')
        mc_connection = f'{hv_conn_type}_{lv_conn_type}'

        trafo_rows.append({
            'CKT_KEY': circuit_name,
            'TRAFO_IDX': t_idx,
            'HV_BUS': hv_bus,
            'LV_BUS': lv_bus,
            'HV_BUS_VN_KV': hv_vn_kv,
            'LV_BUS_VN_KV': lv_vn_kv,
            'MC_HV_WINDING_KV': hv_winding,
            'MC_LV_WINDING_KV': lv_winding,
            'MC_CONNECTION (derived)': mc_connection,
            'CYME_CONNECTION': cyme_connection,
            'CONNECTION_MATCH': mc_connection.replace('_', '') == cyme_connection.replace('_', '').replace('g', ''),
            'MC_FROM_PHASES': [PHASE_INT_TO_LETTER.get(int(p), '?') for p in from_phases],
            'MC_TO_PHASES': [PHASE_INT_TO_LETTER.get(int(p), '?') for p in to_phases],
            'MC_SN_MVA_PER_CIRCUIT': sn_mva,
            'MC_TOTAL_SN_MVA': mc_total_sn_mva,
            'MC_VK_PCT (per side)': vk_pct,
            'MC_VKR_PCT (per side)': vkr_pct,
            'CYME_Z1_PCT (fed as-is from mc)': cyme_z1_pct,
            'CYME_X1R1_RATIO': cyme_x1r1,
            'CYME_PrimaryV (L-L)': hv_vn_kv,
            'CYME_SecondaryV (L-L)': lv_vn_kv,
            'CYME_RatingKVA': cyme_rating_kva,
            'MC_TAP_POS': tap_pos,
            'N_CIRCUITS': len(flat),
        })

    df_trafos = pd.DataFrame(trafo_rows)
    return df_buses, df_trafos


# --- Run for all validation circuits ---
pkl_dir = Path(r"c:\Users\fgonzales\git\mc-0.0.1.15\networks_final")
pkl_files = sorted(pkl_dir.glob("*.pkl"))
total_files = len(pkl_files)

bus_comparison_all = []
trafo_comparison_all = []

t_start = time.time()
for i, pkl_file in enumerate(pkl_files, 1):
    circuit_key = pkl_file.stem
    t0 = time.time()
    with open(pkl_file, 'rb') as fh:
        net = pickle.load(fh)

    df_b, df_t = compare_bus_voltages_and_transformers(net, circuit_key)
    bus_comparison_all.append(df_b)
    trafo_comparison_all.append(df_t)

    elapsed = time.time() - t0
    total_elapsed = time.time() - t_start
    avg_per = total_elapsed / i
    eta = avg_per * (total_files - i)
    print(f"[{i}/{total_files}] {circuit_key}: {len(df_b)} buses, {len(df_t)} trafos  "
          f"({elapsed:.1f}s, ETA {eta:.0f}s)")

df_all_buses = pd.concat(bus_comparison_all, ignore_index=True) if bus_comparison_all else pd.DataFrame()
df_all_trafos = pd.concat(trafo_comparison_all, ignore_index=True) if trafo_comparison_all else pd.DataFrame()

print(f"\nDone in {time.time() - t_start:.1f}s — "
      f"{len(df_all_buses)} bus entries, {len(df_all_trafos)} transformer entries across {len(bus_comparison_all)} circuits")

In [ ]:
# --- Show buses where CYME would NOT have an explicit base voltage ---
# These are buses that are neither source nor transformer LV buses.
# CYME inherits voltage from upstream; mc always uses explicit vn_kv.
unset_buses = df_all_buses[df_all_buses['CYME_UserDefinedBaseVoltage'] == 'inherited/unset']
print(f"Buses WITHOUT explicit CYME UserDefinedBaseVoltage: {len(unset_buses)} / {len(df_all_buses)}")
print(f"Buses WITH explicit CYME voltage (source + trafo LV): {len(df_all_buses) - len(unset_buses)}")
print()

# Show per-circuit summary
voltage_summary = df_all_buses.groupby('CKT_KEY').agg(
    n_buses=('BUS_IDX', 'count'),
    n_source=('IS_SOURCE', 'sum'),
    n_trafo_lv=('IS_TRAFO_LV', 'sum'),
    n_cyme_unset=('CYME_UserDefinedBaseVoltage', lambda x: (x == 'inherited/unset').sum()),
    voltage_levels=('MC_VN_KV (L-L)', lambda x: sorted(x.unique().tolist())),
).reset_index()
voltage_summary

In [ ]:
# --- Transformer configuration comparison ---
print("=== TRANSFORMER CONNECTION TYPE COMPARISON ===\n")

if not df_all_trafos.empty:
    # Show connection mismatches (mc derived vs CYME simplified)
    mismatches = df_all_trafos[df_all_trafos['CONNECTION_MATCH'] == False]
    print(f"Connection type mismatches: {len(mismatches)} / {len(df_all_trafos)} transformers")
    if not mismatches.empty:
        print("\nMismatched transformers:")
        print(mismatches[['CKT_KEY', 'TRAFO_IDX', 'MC_CONNECTION (derived)', 'CYME_CONNECTION',
                          'MC_FROM_PHASES', 'MC_TO_PHASES']].to_string(index=False))

    print(f"\n\n=== CYME CONNECTION DISTRIBUTION ===")
    print(df_all_trafos['CYME_CONNECTION'].value_counts())
    print(f"\n=== MC CONNECTION DISTRIBUTION ===")
    print(df_all_trafos['MC_CONNECTION (derived)'].value_counts())

    # Impedance comparison: mc stores half-impedance per side,
    # but mc_to_cyme.py feeds vk_percent directly as CYME Z1%.
    # CYME expects TOTAL positive-sequence impedance.
    # If mc has vk/2 and CYME gets vk/2 as "total", the impedance is HALVED.
    print(f"\n\n=== IMPEDANCE ANALYSIS ===")
    print("MC stores vk_percent as HALF the total (split between HV and LV sides).")
    print("mc_to_cyme.py feeds this half-value directly to CYME as PositiveSequenceImpedancePercent.")
    print("This means CYME sees HALF the actual transformer impedance!\n")

    imp_summary = df_all_trafos[['CKT_KEY', 'TRAFO_IDX', 'MC_VK_PCT (per side)',
                                  'CYME_Z1_PCT (fed as-is from mc)', 'MC_TOTAL_SN_MVA', 'CYME_RatingKVA']].copy()
    imp_summary['CORRECT_TOTAL_Z1_PCT'] = imp_summary['MC_VK_PCT (per side)'] * 2
    imp_summary['Z_ERROR_FACTOR'] = 0.5  # CYME gets half the impedance
    imp_summary
else:
    print("No transformers found in the loaded networks.")

In [ ]:
# --- Voltage discrepancy analysis: buses with different voltage levels ---
# Check if any buses have a vn_kv that doesn't match what you'd expect
# from the network topology (e.g., all buses on LV side of a 12kV/4kV trafo should be 4kV)

print("=== BUS VOLTAGE LEVEL ANALYSIS ===\n")

for ckt in df_all_buses['CKT_KEY'].unique():
    ckt_buses = df_all_buses[df_all_buses['CKT_KEY'] == ckt]
    vlevels = sorted(ckt_buses['MC_VN_KV (L-L)'].unique())
    source_v = ckt_buses[ckt_buses['IS_SOURCE']]['MC_VN_KV (L-L)'].values
    trafo_lv_buses = ckt_buses[ckt_buses['IS_TRAFO_LV']]

    print(f"--- {ckt} ---")
    print(f"  Voltage levels: {vlevels}")
    print(f"  Source voltage: {source_v}")
    for _, tb in trafo_lv_buses.iterrows():
        print(f"  Trafo LV bus {tb['BUS_NAME']} (idx {tb['BUS_IDX']}): vn_kv={tb['MC_VN_KV (L-L)']} kV  [{tb['TRAFO_NAME']}]")

    # Flag buses where vn_kv doesn't match the expected zone
    # (i.e. bus is not source/trafo-LV but has a different voltage than its zone)
    non_special = ckt_buses[~ckt_buses['IS_SOURCE'] & ~ckt_buses['IS_TRAFO_LV']]
    if not non_special.empty:
        vcount = non_special['MC_VN_KV (L-L)'].value_counts()
        print(f"  Regular buses by voltage: {dict(vcount)}")
    print()

In [ ]:
# --- Full transformer table for inspection ---
if not df_all_trafos.empty:
    display_cols = [
        'CKT_KEY', 'TRAFO_IDX', 'HV_BUS', 'LV_BUS',
        'HV_BUS_VN_KV', 'LV_BUS_VN_KV',
        'MC_HV_WINDING_KV', 'MC_LV_WINDING_KV',
        'MC_CONNECTION (derived)', 'CYME_CONNECTION', 'CONNECTION_MATCH',
        'MC_FROM_PHASES', 'MC_TO_PHASES',
        'MC_SN_MVA_PER_CIRCUIT', 'MC_TOTAL_SN_MVA',
        'MC_VK_PCT (per side)', 'MC_VKR_PCT (per side)',
        'CYME_Z1_PCT (fed as-is from mc)', 'CYME_X1R1_RATIO',
        'CYME_RatingKVA', 'MC_TAP_POS',
    ]
    df_all_trafos[display_cols]

In [ ]:
# --- Power flow settings comparison: key differences that affect results ---
print("=" * 80)
print("POWER FLOW SETTINGS COMPARISON: MC vs CYME")
print("=" * 80)

differences = [
    ("Line Charging",
     "INCLUDED in Y_network admittance matrix",
     "DISABLED (IncludeLineCharging=0)",
     "CYME ignores shunt capacitance of lines → lower reactive power injection → different voltage profile"),

    ("Voltage-Sensitive Load Model",
     "Supports const_z_percent and const_i_percent per load (ZIP model)",
     "DISABLED (VoltageSensitivityLoadModel.V=0 → constant P/Q only)",
     "MC loads may have voltage-dependent behavior; CYME treats all as constant PQ"),

    ("Source Impedance",
     "Included via ext_grid_sequence R/X values in Y_source",
     "DISABLED (IncludeSourceImpedance=0)",
     "CYME treats source as ideal → source bus voltage is exact setpoint"),

    ("Line Transposition",
     "Full phase-unbalanced matrices (no transposition)",
     "DISABLED (AssumeLineTransposition=0)",
     "Both use untranspositioned lines — this is consistent"),

    ("Transformer Impedance",
     "vk_percent stored as HALF total (split HV/LV sides)",
     "Receives vk_percent directly as PositiveSequenceImpedancePercent",
     "⚠️ POSSIBLE BUG: CYME may be getting half the correct impedance if it expects total Z%"),

    ("Transformer Connection",
     "Full phase mapping via from_phase/to_phase arrays per circuit",
     "Simplified to D_Yg or Yg_Yg based on presence of non-zero to_phase",
     "Connection logic may not capture all vector group variations correctly"),

    ("Capacitor/Regulator Controls",
     "Can run iterative control loop (run_control=True)",
     "ALL controls LOCKED by default (LockSwitchedCapacitors=1, etc.)",
     "CYME locks cap banks and regulators at initial state"),

    ("Bus Base Voltage Assignment",
     "Explicit vn_kv per bus in net.bus table (set at creation time)",
     "UserDefinedBaseVoltage set only on source and transformer LV nodes; rest inherited",
     "CYME engine propagates voltage through topology; MC relies on explicit assignment"),
]

for name, mc_val, cyme_val, impact in differences:
    print(f"\n{'─' * 80}")
    print(f"  {name}")
    print(f"  MC:   {mc_val}")
    print(f"  CYME: {cyme_val}")
    print(f"  Impact: {impact}")

In [ ]:
# --- Display full bus table with voltage comparison ---
df_all_buses